# Thư viện

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Gộp file

In [14]:
import pandas as pd
import json
import glob
import os

# 1. ĐƯỜNG DẪN THƯ MỤC CHỨA CÁC FILE JSON
folder_path = r"D:\PYTHON\Data mining\VN-IT-Job-Mining\data\raw\vietnamworks\2026-10-20"

# Tìm toàn bộ các file có đuôi .json trong thư mục này
json_files = glob.glob(os.path.join(folder_path, "*.json"))

# Khởi tạo list để chứa dữ liệu từ tất cả các file
raw_payloads = []

# 2. VÒNG LẶP ĐỌC VÀ GỘP DỮ LIỆU
print(f"Đang tiến hành đọc {len(json_files)} file JSON...")

for file in json_files:
    try:
        with open(file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            # Chỉ trích xuất node 'raw_payload' vì đây là nơi chứa các cột cốt lõi (jobTitle, salary,...)
            if 'raw_payload' in data:
                raw_payloads.append(data['raw_payload'])
            else:
                # Fallback: Nếu file nào không có raw_payload, lấy toàn bộ data
                raw_payloads.append(data)
                
    except Exception as e:
        print(f"Lỗi khi đọc file {file}: {e}")

# 3. CHUYỂN ĐỔI THÀNH DATAFRAME
# pd.json_normalize sẽ trải phẳng (flatten) các object JSON thành các cột 
# và GIỮ NGUYÊN 100% dữ liệu thô ban đầu, không thay đổi hay xóa bất kỳ giá trị nào.
df_vietnamworks = pd.json_normalize(raw_payloads)

# Kiểm tra kết quả
print("\nĐã gộp thành công!")
print(f"Số lượng bản ghi (dòng): {df_vietnamworks.shape[0]}")
print(f"Số lượng trường dữ liệu (cột): {df_vietnamworks.shape[1]}")

# (Tùy chọn) Xem thử 3 dòng đầu tiên và các cột hiện có
display(df_vietnamworks.head(3))

Đang tiến hành đọc 1128 file JSON...

Đã gộp thành công!
Số lượng bản ghi (dòng): 1128
Số lượng trường dữ liệu (cột): 130


,jobId,jobTitle,jobUrl,userId,companyLogo,companyUrl,alias,createdOn,approvedOn,approvedOnText,...,job_info.NGÀNH NGHỀ,job_info.KỸ NĂNG,job_info.LĨNH VỰC,job_info.NGÔN NGỮ TRÌNH BÀY HỒ SƠ,job_info.SỐ NĂM KINH NGHIỆM TỐI THIỂU,job_info.QUỐC TỊCH,company.name,company.address,company.contact,company.culture_url
0,1811606,Sales Engineer,https://www.vietnamworks.com/sales-engineer-17...,7646602,https://images.vietnamworks.com/pictureofcompa...,,sales-engineer-1790752331072092706,2024-08-15T14:32:01+07:00,2026-09-30T14:12:11+07:00,Đăng 20 ngày trước,...,"{'parentId': 21, 'parentName': 'Sales', 'paren...","Communication skills, Customer Management, Inv...",IT,Tiếng Anh,2,Không giới hạn,MELCO INDUSTRIAL SUPPLIES (VIETNAM) COMPANY LI...,"No. 30A, Street No.10, Quarter 2, Hiep Binh Ward",Casey Hiu,
1,2025655,Junior ISO Consultant,https://www.vietnamworks.com/junior-iso-consul...,3443935,https://images.vietnamworks.com/pictureofcompa...,,junior-iso-consultant-1791534347020006155,2026-03-18T15:10:03+07:00,2026-10-09T15:25:47+07:00,Đăng 11 ngày trước,...,"{'parentId': 5, 'parentName': 'Information Tec...","Quản Trị Rủi Ro Thông Tin, ISO Standards, ISO ...",IT,Tiếng Việt,1,Không giới hạn,Công Ty CP Đầu Tư Và Phát Triển Công Nghệ Tabviet,"Tòa nhà Phương Nam, 219 Quốc lộ 13, Phường 26,...",Team Hr,
2,2071836,Kỹ Sư Thiết Kế Cơ Khí Tự Động Hóa (Automation ...,https://www.vietnamworks.com/ky-su-thiet-ke-co...,8217573,https://images.vietnamworks.com/pictureofcompa...,,ky-su-thiet-ke-co-khi-tu-djong-hoa-automation-...,2026-06-23T10:09:44+07:00,2026-09-25T11:35:56+07:00,Đăng 25 ngày trước,...,"{'parentId': 7, 'parentName': 'Design', 'paren...","Tự Động Hóa, Thiết Kế Cơ Khí, IronCAD, Điện tử...",IT,Bất kỳ,2,Không giới hạn,Công ty TNHH cowaFA,"Lô số 1 ô số 3 điểm công nghiệp xã Di Trạch, x...",HR,


# Trụ Cột 1: Độ Hoàn Thiện (Completeness & Missing Value Profiling)

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ==========================================
# 1. HÀM QUÉT VÀ LÀM SẠCH NULL MARKERS ẨN
# ==========================================
def clean_hidden_nulls(df):
    """
    Nhận diện và chuyển đổi các giá trị khuyết thiếu dạng text hoặc mảng rỗng về chuẩn np.nan
    """
    df_cleaned = df.copy()
    
    # 1.1. Xử lý các null markers dạng chuỗi
    null_markers = [
        "", " ", "N/A", "n/a", "null", "Null", "NULL", "None", "none",
        "Chưa cập nhật", "Đang cập nhật", "Thỏa thuận", "Thoả thuận", "Thương lượng"
    ]
    
    for col in df_cleaned.select_dtypes(include=['object']):
        df_cleaned[col] = df_cleaned[col].apply(
            lambda x: np.nan if str(x).strip() in null_markers else x
        )
    
    # 1.2. Xử lý các list/dict rỗng (ví dụ: requirements_arr: [])
    # Sử dụng map (với pandas >= 2.1.0) hoặc applymap để quét toàn bộ Dataframe
    df_cleaned = df_cleaned.map(
        lambda x: np.nan if isinstance(x, (list, dict)) and len(x) == 0 else x
    )
    
    return df_cleaned

# ==========================================
# 2. HÀM TRỰC QUAN HÓA: BAR CHART KHUYẾT THIẾU
# ==========================================
def plot_completeness(df, save_path):
    """
    Vẽ Bar Chart ngang thể hiện tỷ lệ khuyết thiếu của từng trường.
    Chỉ hiển thị các cột có dữ liệu bị thiếu.
    """
    # Tính tỷ lệ % null
    null_percentages = (df.isnull().sum() / len(df)) * 100
    
    # Chỉ lấy các cột bị khuyết thiếu (Tỷ lệ > 0)
    cols_with_nulls = null_percentages[null_percentages > 0]
    
    if cols_with_nulls.empty:
        print("✅ Dữ liệu hoàn hảo, không có cột nào bị khuyết thiếu!")
        return

    # Sắp xếp để thanh dài nhất (tỷ lệ thiếu cao nhất) nằm trên cùng
    missing_pct_sorted = cols_with_nulls.sort_values(ascending=True)

    # Tự động điều chỉnh chiều cao ảnh: mỗi cột chiếm 0.5 inch, tối thiểu là 6 inch
    fig_height = max(6, len(missing_pct_sorted) * 0.5)
    plt.figure(figsize=(12, fig_height))
    
    # Vẽ biểu đồ thanh ngang
    bars = plt.barh(missing_pct_sorted.index, missing_pct_sorted.values, color="#e76f51", alpha=0.85)
    
    # Tùy chỉnh tiêu đề và trục
    plt.title("Tỷ Lệ Khuyết Thiếu Từng Trường (%)", fontsize=15, pad=20, fontweight='bold')
    plt.xlabel("Tỷ lệ Null (%)", fontsize=12)
    plt.yticks(fontsize=11)
    
    # Mở rộng không gian trục X để text % không bị lẹm ra ngoài
    plt.xlim(0, max(missing_pct_sorted.values) + 15) 
    
    # In trực tiếp tỷ lệ % cạnh mỗi thanh bar
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 1, 
                 bar.get_y() + bar.get_height()/2, 
                 f'{width:.1f}%', 
                 va='center', ha='left', fontsize=11, fontweight='bold', color="#264653")
    
    # Tự động căn lề và xử lý layout
    plt.tight_layout()
    
    # Lưu file chuẩn DPI 300, dùng bbox_inches='tight' để không cắt mất tên cột dài
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Hoàn tất! Đã lưu biểu đồ rõ nét tại: {save_path}")

# ==========================================
# 3. HÀM THỐNG KÊ BẢNG DỮ LIỆU
# ==========================================
def missing_value_summary(df):
    """
    Trích xuất bảng thống kê chi tiết số lượng và % khuyết thiếu
    """
    missing_counts = df.isnull().sum()
    missing_percentages = (missing_counts / len(df)) * 100
    
    summary_df = pd.DataFrame({
        'Missing_Count': missing_counts,
        'Missing_Percentage (%)': missing_percentages
    }).sort_values(by='Missing_Percentage (%)', ascending=False)
    
    return summary_df[summary_df['Missing_Count'] > 0]

# ==========================================
# 4. CÁCH GỌI HÀM (THỰC THI PIPELINE)
# ==========================================
# Giả định bạn đã có DataFrame từ bước đọc file JSON: df_vietnamworks

print("Đang làm sạch các giá trị Null ẩn và mảng rỗng...")
df_vw_cleaned = clean_hidden_nulls(df_vietnamworks)

# (Tùy chọn) Lọc bớt cột nếu JSON trả về quá nhiều dữ liệu không cần thiết
# core_columns = ['jobTitle', 'salary', 'skills', ...]
# df_vw_cleaned = df_vw_cleaned[core_columns]

# Xem bảng thống kê
print("\nBảng thống kê khuyết thiếu:")
display(missing_value_summary(df_vw_cleaned))

# Vẽ và xuất file ảnh
save_path = r"D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vietnamworks\01_data_completeness_bar.png"
plot_completeness(df_vw_cleaned, save_path)

Đang làm sạch các giá trị Null ẩn và mảng rỗng...

Bảng thống kê khuyết thiếu:


,Missing_Count,Missing_Percentage (%)
locations,1128,100.000000
companyUrl,1128,100.000000
summary,1128,100.000000
company.culture_url,1128,100.000000
emailAddress,1128,100.000000
nearestGeoLoc,1128,100.000000
redirectTo,1122,99.468085
ageMin,903,80.053191
ageMax,903,80.053191
prettySalary,801,71.010638


✅ Hoàn tất! Đã lưu biểu đồ rõ nét tại: D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vietnamworks\01_data_completeness_bar.png


# Trụ Cột 2: Tính Hợp Lệ & Kiểu Dữ Liệu (Validity & Schema Types)

In [19]:
import re

# ==========================================
# 1. HÀM CHẨN ĐOÁN CHUNG VỀ KIỂU DỮ LIỆU (DTYPES)
# ==========================================
def check_data_types(df):
    """
    Kiểm tra kiểu dữ liệu thực tế của các cột và đối chiếu sơ bộ
    """
    print("--- 1. TỔNG QUAN KIỂU DỮ LIỆU ---")
    dtype_summary = pd.DataFrame(df.dtypes, columns=['Thực_tế_Dtype'])
    dtype_summary['Số_lượng_Non_Null'] = df.notnull().sum()
    dtype_summary['Ví_dụ_Giá_trị'] = df.apply(lambda x: x.dropna().iloc[0] if x.notna().any() else np.nan)
    display(dtype_summary.head(15)) # Hiển thị 15 cột đầu tiên để tránh dài dòng

# ==========================================
# 2. HÀM CHẨN ĐOÁN CỘT NGÀY THÁNG (ISO 8601 vs Chuỗi Tương Đối)
# ==========================================
def validate_datetime_columns(df):
    """
    Kiểm tra xem các cột ngày tháng có đúng chuẩn ISO 8601 hay bị lẫn chuỗi text.
    """
    print("\n--- 2. CHẨN ĐOÁN TÍNH HỢP LỆ NGÀY THÁNG ---")
    date_cols = ['createdOn', 'approvedOn', 'expiredOn']
    
    # Định dạng ISO 8601 Regex (Ví dụ: 2024-08-15T14:32:01+07:00)
    iso_pattern = r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?(?:Z|[+-]\d{2}:\d{2})$'
    
    for col in date_cols:
        if col in df.columns:
            # Lọc bỏ giá trị null để kiểm tra chuỗi
            valid_values = df[col].dropna().astype(str)
            
            # Đếm số lượng đúng/sai format
            is_iso = valid_values.str.match(iso_pattern)
            valid_count = is_iso.sum()
            invalid_count = len(valid_values) - valid_count
            
            print(f"Cột [{col}]:")
            print(f"  - Đúng chuẩn ISO 8601: {valid_count} bản ghi")
            if invalid_count > 0:
                invalid_samples = valid_values[~is_iso].head(3).tolist()
                print(f"  - ⚠️ SAI ĐỊNH DẠNG (Chuỗi tương đối/Lỗi): {invalid_count} bản ghi. Ví dụ: {invalid_samples}")
            else:
                print("  - Trạng thái: Hợp lệ 100%")

# ==========================================
# 3. HÀM CHẨN ĐOÁN CỘT SỐ HỌC (BỊ STRING HÓA/DÍNH KÝ TỰ)
# ==========================================
def validate_numeric_strings(df):
    """
    Phát hiện các cột lẽ ra là số (Lương) nhưng lại chứa ký tự text (VND, tr, tháng).
    """
    print("\n--- 3. CHẨN ĐOÁN SỐ HỌC BỊ STRING HÓA ---")
    
    # Cột lương gốc từ JSON mẫu là chuỗi phức hợp
    target_col = 'prettySalary' 
    if target_col in df.columns:
        samples = df[target_col].dropna().head(3).tolist()
        print(f"Cột [{target_col}] đang lưu dưới dạng {df[target_col].dtype}:")
        print(f"  - Ví dụ giá trị thực tế: {samples}")
        print("  - ⚠️ Chẩn đoán: Đây là cột string phức hợp chứa cả số và text (tr, ₫/tháng, thỏa thuận). Cần bóc tách trong quá trình Pre-processing.")

    # Cột lương Min/Max
    for col in ['salaryMin', 'salaryMax']:
        if col in df.columns:
            dtype_col = df[col].dtype
            print(f"Cột [{col}] đang lưu dưới dạng {dtype_col}:")
            if pd.api.types.is_numeric_dtype(dtype_col):
                print("  - Trạng thái: Hợp lệ (Là dạng số nguyên/thực).")
            else:
                print(f"  - ⚠️ Lỗi: Bị ép kiểu thành Object (chuỗi) do chứa ký tự lạ. Ví dụ: {df[col].dropna().head(2).tolist()}")

# ==========================================
# 4. HÀM CHẨN ĐOÁN CẤU TRÚC MẢNG (LIST/DICT TRONG CỘT)
# ==========================================
def validate_complex_structures(df):
    """
    Kiểm tra các cột chứa cấu trúc List/Dict xem có đúng là list thực sự 
    hay bị string hóa thành "['A', 'B']" (chuỗi).
    """
    print("\n--- 4. CHẨN ĐOÁN CẤU TRÚC MẢNG (SKILLS, BENEFITS) ---")
    array_cols = ['skills', 'benefits', 'workingLocations']
    
    for col in array_cols:
        if col in df.columns:
            sample_val = df[col].dropna().iloc[0]
            actual_type = type(sample_val)
            
            print(f"Cột [{col}]:")
            if actual_type == list:
                # Kiểm tra tiếp cấu trúc bên trong (list chứa chuỗi hay list chứa dict)
                inner_type = type(sample_val[0]) if len(sample_val) > 0 else "Empty"
                print(f"  - Kiểu dữ liệu bề mặt: List (Hợp lệ)")
                print(f"  - Cấu trúc bên trong: List of {inner_type}")
                if inner_type == dict:
                    print("  - ⚠️ Lưu ý Pre-processing: Đây là mảng chứa dictionary/JSON lồng nhau, cần trích xuất khóa (ví dụ: 'skillName') để tạo list[str] phẳng.")
            elif actual_type == str:
                print(f"  - ⚠️ LỖI NGHIÊM TRỌNG: Mảng đã bị string hóa thành {actual_type}. Cần dùng ast.literal_eval() ở bước xử lý.")

# ==========================================
# THỰC THI PIPELINE CHẨN ĐOÁN (Gọi hàm)
# Giả định df_vietnamworks là DataFrame bạn đã đọc từ bước trước
# ==========================================
# Bạn có thể truyền thẳng df_vietnamworks thô vào đây để kiểm tra hiện trạng gốc
print("BẮT ĐẦU CHẨN ĐOÁN TRỤ CỘT 2: TÍNH HỢP LỆ DỮ LIỆU (VIETNAMWORKS)\n" + "="*60)
check_data_types(df_vietnamworks)
validate_datetime_columns(df_vietnamworks)
validate_numeric_strings(df_vietnamworks)
validate_complex_structures(df_vietnamworks)

BẮT ĐẦU CHẨN ĐOÁN TRỤ CỘT 2: TÍNH HỢP LỆ DỮ LIỆU (VIETNAMWORKS)
--- 1. TỔNG QUAN KIỂU DỮ LIỆU ---


,Thực_tế_Dtype,Số_lượng_Non_Null,Ví_dụ_Giá_trị
jobId,int64,1128,1811606
jobTitle,object,1128,Sales Engineer
jobUrl,object,1128,https://www.vietnamworks.com/sales-engineer-17...
userId,int64,1128,7646602
companyLogo,object,1128,https://images.vietnamworks.com/pictureofcompa...
companyUrl,object,1128,
alias,object,1128,sales-engineer-1790752331072092706
createdOn,object,1128,2024-08-15T14:32:01+07:00
approvedOn,object,1128,2026-09-30T14:12:11+07:00
approvedOnText,object,1128,Đăng 20 ngày trước



--- 2. CHẨN ĐOÁN TÍNH HỢP LỆ NGÀY THÁNG ---
Cột [createdOn]:
  - Đúng chuẩn ISO 8601: 1128 bản ghi
  - Trạng thái: Hợp lệ 100%
Cột [approvedOn]:
  - Đúng chuẩn ISO 8601: 1128 bản ghi
  - Trạng thái: Hợp lệ 100%
Cột [expiredOn]:
  - Đúng chuẩn ISO 8601: 1128 bản ghi
  - Trạng thái: Hợp lệ 100%

--- 3. CHẨN ĐOÁN SỐ HỌC BỊ STRING HÓA ---
Cột [prettySalary] đang lưu dưới dạng object:
  - Ví dụ giá trị thực tế: ['8tr-18tr ₫/tháng', '$ 7tr-9tr /tháng', '17tr-30tr ₫/tháng']
  - ⚠️ Chẩn đoán: Đây là cột string phức hợp chứa cả số và text (tr, ₫/tháng, thỏa thuận). Cần bóc tách trong quá trình Pre-processing.
Cột [salaryMin] đang lưu dưới dạng int64:
  - Trạng thái: Hợp lệ (Là dạng số nguyên/thực).
Cột [salaryMax] đang lưu dưới dạng int64:
  - Trạng thái: Hợp lệ (Là dạng số nguyên/thực).

--- 4. CHẨN ĐOÁN CẤU TRÚC MẢNG (SKILLS, BENEFITS) ---
Cột [skills]:
  - Kiểu dữ liệu bề mặt: List (Hợp lệ)
  - Cấu trúc bên trong: List of <class 'dict'>
  - ⚠️ Lưu ý Pre-processing: Đây là mảng chứa dictiona

# biểu đồ 2:

In [20]:
def plot_salary_distribution(df, save_path):
    """
    Vẽ 2 biểu đồ: Phân phối lương gốc (Triệu VNĐ) và Phân phối sau khi biến đổi Log.
    Tính toán và hiển thị độ lệch (Skewness), Mean, Median.
    """
    # 1. TIỀN XỬ LÝ & LÀM SẠCH CỘT LƯƠNG
    if 'salaryMax' not in df.columns:
        print("❌ Không tìm thấy cột 'salaryMax' trong DataFrame.")
        return
        
    # Lọc bỏ null và các giá trị <= 0 (Nhà tuyển dụng để lương 'Thỏa thuận' thường được hệ thống gán bằng 0)
    valid_salaries = df[df['salaryMax'] > 0]['salaryMax'].dropna()
    
    if valid_salaries.empty:
        print("❌ Không có dữ liệu lương hợp lệ (hoặc toàn bộ là 0) để vẽ biểu đồ.")
        return
        
    # Quy đổi từ VNĐ sang Triệu VNĐ (Ví dụ: 18000000 -> 18.0)
    salaries_millions = valid_salaries / 1_000_000
    
    # 2. KHỞI TẠO KHUNG BIỂU ĐỒ (1 dòng, 2 cột)
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # ==========================================
    # BIỂU ĐỒ TRÁI: PHÂN PHỐI LƯƠNG GỐC
    # ==========================================
    sns.histplot(salaries_millions, kde=True, ax=axes[0], color="#1d3557", bins=30)
    
    # Tính toán các chỉ số thống kê
    median_val = salaries_millions.median()
    mean_val = salaries_millions.mean()
    skewness_val = salaries_millions.skew()
    
    # Vẽ đường trung bình và trung vị
    axes[0].axvline(median_val, color="red", linestyle="--", linewidth=2, label=f"Median: {median_val:.1f}M")
    axes[0].axvline(mean_val, color="orange", linestyle="-", linewidth=2, label=f"Mean: {mean_val:.1f}M")
    
    axes[0].set_title(f"Phân Phối Lương Gốc (Skewness = {skewness_val:.2f})", fontsize=14, pad=15)
    axes[0].set_xlabel("Triệu VNĐ/tháng", fontsize=12)
    axes[0].set_ylabel("Số lượng tin tuyển dụng", fontsize=12)
    axes[0].legend(fontsize=11)
    
    # ==========================================
    # BIỂU ĐỒ PHẢI: LOG-TRANSFORMED (BÌNH THƯỜNG HÓA DỮ LIỆU)
    # ==========================================
    # Sử dụng log1p = log(1 + x) để xử lý an toàn
    log_salaries = np.log1p(salaries_millions)
    log_skewness = log_salaries.skew()
    
    sns.histplot(log_salaries, kde=True, ax=axes[1], color="#e76f51", bins=30)
    axes[1].set_title(f"Phân Phối Sau Log-Transform (Skewness = {log_skewness:.2f})", fontsize=14, pad=15)
    axes[1].set_xlabel("log(1 + Lương [Triệu VNĐ])", fontsize=12)
    axes[1].set_ylabel("Số lượng tin tuyển dụng", fontsize=12)
    
    # 3. CĂN CHỈNH VÀ LƯU FILE
    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    # DPI 300 chuẩn báo cáo
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Hoàn tất! Đã lưu biểu đồ phân phối lương tại: {save_path}")

# ==========================================
# THỰC THI PIPELINE VẼ BIỂU ĐỒ 2
# ==========================================
save_path_chart_2 = r"D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vietnamworks\02_salary_distribution_skewness.png"
plot_salary_distribution(df_vietnamworks, save_path_chart_2)

✅ Hoàn tất! Đã lưu biểu đồ phân phối lương tại: D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vietnamworks\02_salary_distribution_skewness.png


# Trụ cột 3: Tính Duy Nhất & Trùng Lặp (Uniqueness & Duplication)

In [21]:
import pandas as pd

def check_duplication(df):
    """
    Hàm đo lường tỷ lệ trùng lặp dữ liệu theo 3 cấp độ:
    1. Trùng lặp tuyệt đối (Exact duplicates)
    2. Trùng lặp khóa định danh (ID duplicates)
    3. Trùng lặp logic (Semantic duplicates)
    """
    print("--- TRỤ CỘT 3: CHẨN ĐOÁN TÍNH DUY NHẤT & TRÙNG LẶP ---")
    total_records = len(df)
    print(f"Tổng số bản ghi ban đầu: {total_records}")

    if total_records == 0:
        print("❌ DataFrame rỗng, không thể kiểm tra trùng lặp.")
        return

    # ==========================================
    # 1. TRÙNG LẶP TUYỆT ĐỐI (100% CÁC CỘT GIỐNG NHAU)
    # ==========================================
    # Ép toàn bộ DataFrame về dạng chuỗi (string) để tránh lỗi unhashable type do chứa list/dict
    df_string_format = df.astype(str)
    exact_dups = df_string_format.duplicated().sum()
    exact_pct = (exact_dups / total_records) * 100
    
    print(f"\n1. Trùng lặp tuyệt đối (Exact Duplicates):")
    print(f"   - Số lượng: {exact_dups} bản ghi")
    print(f"   - Tỷ lệ: {exact_pct:.2f}%")

    # ==========================================
    # 2. TRÙNG LẶP KHÓA ĐỊNH DANH (ID / URL)
    # ==========================================
    # Dựa vào file mẫu, VietnamWorks sử dụng 'jobId' và 'jobUrl'
    id_col = 'jobId' if 'jobId' in df.columns else 'jobUrl'
    
    if id_col in df.columns:
        id_dups = df.duplicated(subset=[id_col]).sum()
        id_pct = (id_dups / total_records) * 100
        print(f"\n2. Trùng lặp khóa định danh (Cột '{id_col}'):")
        print(f"   - Số lượng: {id_dups} bản ghi")
        print(f"   - Tỷ lệ: {id_pct:.2f}%")
    else:
        id_dups = 0
        print("\n2. Trùng lặp khóa định danh: Không tìm thấy cột 'jobId' hay 'jobUrl' trong dữ liệu.")

    # ==========================================
    # 3. TRÙNG LẶP LOGIC (COMPANY + TITLE + LOCATION)
    # ==========================================
    logical_cols = ['jobTitle', 'companyName', 'location']
    existing_logical_cols = [c for c in logical_cols if c in df.columns]
    
    if len(existing_logical_cols) == len(logical_cols):
        # Chuẩn hóa (lower-case & strip) để phát hiện HR đăng lại tin nhưng gõ khác in hoa/in thường
        temp_df = df[logical_cols].fillna('').astype(str).apply(lambda x: x.str.lower().str.strip())
        logical_dups = temp_df.duplicated().sum()
        logical_pct = (logical_dups / total_records) * 100
        
        print(f"\n3. Trùng lặp logic (Theo {' + '.join(logical_cols)}):")
        print(f"   - Số lượng: {logical_dups} bản ghi")
        print(f"   - Tỷ lệ: {logical_pct:.2f}%")
    else:
        logical_dups = 0
        print(f"\n3. Trùng lặp logic: Bỏ qua do thiếu một trong các cột {logical_cols}.")

    # ==========================================
    # KẾT LUẬN & ĐỀ XUẤT PRE-PROCESSING
    # ==========================================
    print("\n" + "="*50)
    max_dups = max(exact_dups, id_dups, logical_dups)
    if max_dups > 0:
        print("⚠️ CHẨN ĐOÁN: Dữ liệu chứa bản ghi trùng lặp (Overlapping).")
        print("💡 HƯỚNG XỬ LÝ (Pre-processing):")
        print(f"   -> Bước 1: drop_duplicates(subset=['{id_col}']) để xóa tin cào trùng do lỗi phân trang.")
        print(f"   -> Bước 2: drop_duplicates(subset={logical_cols}) giữ lại tin mới nhất (sort theo 'createdOn') để xóa tin bị HR đăng lặp (Repost).")
    else:
        print("✅ TUYỆT VỜI: Không phát hiện bản ghi trùng lặp nào. Pipeline collection hoạt động hoàn hảo.")

# ==========================================
# THỰC THI PIPELINE CHẨN ĐOÁN TRÙNG LẶP
# ==========================================
check_duplication(df_vietnamworks)

--- TRỤ CỘT 3: CHẨN ĐOÁN TÍNH DUY NHẤT & TRÙNG LẶP ---
Tổng số bản ghi ban đầu: 1128

1. Trùng lặp tuyệt đối (Exact Duplicates):
   - Số lượng: 0 bản ghi
   - Tỷ lệ: 0.00%

2. Trùng lặp khóa định danh (Cột 'jobId'):
   - Số lượng: 0 bản ghi
   - Tỷ lệ: 0.00%

3. Trùng lặp logic (Theo jobTitle + companyName + location):
   - Số lượng: 11 bản ghi
   - Tỷ lệ: 0.98%

⚠️ CHẨN ĐOÁN: Dữ liệu chứa bản ghi trùng lặp (Overlapping).
💡 HƯỚNG XỬ LÝ (Pre-processing):
   -> Bước 1: drop_duplicates(subset=['jobId']) để xóa tin cào trùng do lỗi phân trang.
   -> Bước 2: drop_duplicates(subset=['jobTitle', 'companyName', 'location']) giữ lại tin mới nhất (sort theo 'createdOn') để xóa tin bị HR đăng lặp (Repost).


# Trụ Cột 5: Độ Thiên Lệch & Tính Mất Cân Bằng (Bias & Balance Profiling)

In [ ]:
# ==========================================
# 1. HÀM ĐÁNH GIÁ CHỈ SỐ MẤT CÂN BẰNG (TEXT REPORT)
# ==========================================
def profile_bias_and_balance(df):
    print("--- TRỤ CỘT 5: CHẨN ĐOÁN THIÊN LỆCH & MẤT CÂN BẰNG ---")
    total_records = len(df)
    if total_records == 0:
        return
        
    # 1. Mất cân bằng Cấp bậc (Seniority Bias)
    if 'jobLevelVI' in df.columns:
        print("\n1. Phân bổ Cấp bậc (Seniority):")
        seniority_counts = df['jobLevelVI'].value_counts()
        for level, count in seniority_counts.items():
            pct = (count / total_records) * 100
            print(f"   - {level}: {count} tin ({pct:.1f}%)")
        if len(seniority_counts) >= 2:
            imbalance_ratio = seniority_counts.max() / seniority_counts.min()
            print(f"   => Imbalance Ratio (Max/Min): {imbalance_ratio:.1f}x")

    # 2. Thiên lệch Địa lý (Geographic Bias)
    if 'location' in df.columns:
        print("\n2. Phân bổ Địa lý (Location):")
        geo_counts = df['location'].value_counts().head(5)
        for loc, count in geo_counts.items():
            pct = (count / total_records) * 100
            print(f"   - {loc}: {count} tin ({pct:.1f}%)")

    # 3. Tính Thưa thớt Kỹ năng (Skill Sparsity - Pareto 80/20)
    if 'keywords' in df.columns:
        # Bóc tách mảng kỹ năng thành các dòng đơn lẻ
        all_skills = df['keywords'].dropna().explode().str.strip()
        skill_counts = all_skills.value_counts()
        total_skill_tags = len(all_skills)
        unique_skills = len(skill_counts)
        
        # Tính xem bao nhiêu % kỹ năng top đầu chiếm 80% tổng tần suất
        cumulative_pct = (skill_counts.cumsum() / total_skill_tags) * 100
        top_skills_for_80_pct = cumulative_pct[cumulative_pct <= 80].count()
        pareto_ratio = (top_skills_for_80_pct / unique_skills) * 100
        
        print("\n3. Chẩn đoán Kỹ năng (Quy luật Pareto):")
        print(f"   - Tổng số tags kỹ năng xuất hiện: {total_skill_tags}")
        print(f"   - Số lượng kỹ năng duy nhất (Unique): {unique_skills}")
        print(f"   - Nhóm Top {top_skills_for_80_pct} kỹ năng (chiếm {pareto_ratio:.1f}% unique skills) đang thống trị 80% nhu cầu thị trường.")
        print(f"   - ⚠️ Chẩn đoán: Dữ liệu kỹ năng có tính thưa thớt (Sparsity) cao. Nhiều kỹ năng ngách (long-tail) chỉ xuất hiện 1-2 lần.")

# ==========================================
# 2. HÀM VẼ BIỂU ĐỒ 6: PARETO 80/20 KỸ NĂNG
# ==========================================
def plot_skills_pareto(df, save_path):
    if 'keywords' not in df.columns:
        return
        
    all_skills = df['keywords'].dropna().explode().str.strip()
    top20 = all_skills.value_counts().head(20).reset_index()
    top20.columns = ["Skill", "Frequency"]
    top20["Cum_Percent"] = (top20["Frequency"].cumsum() / top20["Frequency"].sum()) * 100
    
    fig, ax1 = plt.subplots(figsize=(13, 6))
    ax2 = ax1.twinx()
    
    # Vẽ Bar Chart tần suất
    sns.barplot(data=top20, x="Skill", y="Frequency", ax=ax1, palette="viridis", hue="Skill", legend=False)
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha="right", fontsize=10)
    ax1.set_ylabel("Tần suất xuất hiện", fontsize=11)
    
    # Vẽ đường Tích lũy (Cumulative Line)
    sns.lineplot(data=top20, x="Skill", y="Cum_Percent", ax=ax2, color="#d62828", marker="o", linewidth=2.5)
    ax2.axhline(80, color="gray", linestyle="--", alpha=0.7, label="Ngưỡng 80% Pareto")
    ax2.set_ylabel("Tỷ lệ tích lũy (%)", fontsize=11)
    ax2.set_ylim(0, 105)
    ax2.legend(loc="lower right")
    
    plt.title("Phân Phối Tần Suất Kỹ Năng & Kiểm Định Quy Luật Pareto (Top 20)", fontsize=14, fontweight='bold', pad=15)
    plt.tight_layout()
    
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Đã lưu Biểu đồ 6 (Pareto Skills) tại: {save_path}")

# ==========================================
# 3. HÀM VẼ BIỂU ĐỒ 7: CHẨN ĐOÁN MẤT CÂN BẰNG LỚP
# ==========================================
def plot_bias_and_balance(df, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Biểu đồ 1: Mất cân bằng Cấp bậc (Seniority Class Imbalance)
    if 'jobLevelVI' in df.columns:
        seniority_counts = df['jobLevelVI'].value_counts()
        sns.barplot(x=seniority_counts.values, y=seniority_counts.index, ax=axes[0], palette="flare", hue=seniority_counts.index, legend=False)
        
        imbalance_ratio = seniority_counts.max() / seniority_counts.min() if len(seniority_counts) >= 2 else 1
        axes[0].set_title(f"Mất Cân Bằng Cấp Bậc\n(Imbalance Ratio = {imbalance_ratio:.1f}x)", fontsize=13, pad=10)
        axes[0].set_xlabel("Số lượng tin tuyển dụng", fontsize=11)
        axes[0].set_ylabel("")
    
    # Biểu đồ 2: Thiên lệch Địa lý (Donut Chart)
    if 'location' in df.columns:
        top_loc = df['location'].value_counts().head(5)
        # Gộp các tỉnh còn lại thành "Khác" nếu có quá nhiều
        if len(df['location'].value_counts()) > 5:
            other_count = df['location'].value_counts().iloc[5:].sum()
            top_loc['Các tỉnh khác'] = other_count
            
        axes[1].pie(top_loc, labels=top_loc.index, autopct="%1.1f%%", startangle=140,
                    colors=sns.color_palette("Set2"), pctdistance=0.8, 
                    textprops={'fontsize': 11})
        # Biến Pie thành Donut
        centre_circle = plt.Circle((0,0), 0.55, fc='white')
        axes[1].add_artist(centre_circle)
        axes[1].set_title("Thiên Lệch Địa Lý\n(Geographic Bias)", fontsize=13, pad=10)
    
    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Đã lưu Biểu đồ 7 (Bias & Balance) tại: {save_path}")

# ==========================================
# THỰC THI PIPELINE 
# ==========================================
# 1. In báo cáo Text ra console
profile_bias_and_balance(df_vietnamworks)

# 2. Vẽ và xuất biểu đồ
path_chart_6 = r"D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vietnamworks\03_skills_pareto_distribution.png"
path_chart_7 = r"D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vietnamworks\04_bias_and_class_balance.png"

plot_skills_pareto(df_vietnamworks, path_chart_6)
plot_bias_and_balance(df_vietnamworks, path_chart_7)

--- TRỤ CỘT 5: CHẨN ĐOÁN THIÊN LỆCH & MẤT CÂN BẰNG ---

1. Phân bổ Cấp bậc (Seniority):
   - Nhân viên: 905 tin (80.2%)
   - Trưởng phòng: 159 tin (14.1%)
   - Mới Tốt Nghiệp: 43 tin (3.8%)
   - Giám Đốc và Cấp Cao Hơn: 21 tin (1.9%)
   => Imbalance Ratio (Max/Min): 43.1x

2. Phân bổ Địa lý (Location):
   - Ha Noi: 520 tin (46.1%)
   - Ho Chi Minh: 351 tin (31.1%)
   - Bac Ninh: 18 tin (1.6%)
   - Ha Noi, Ho Chi Minh: 17 tin (1.5%)
   - Ho Chi Minh, Ha Noi: 16 tin (1.4%)

3. Chẩn đoán Kỹ năng (Quy luật Pareto):
   - Tổng số tags kỹ năng xuất hiện: 5369
   - Số lượng kỹ năng duy nhất (Unique): 2778
   - Nhóm Top 1704 kỹ năng (chiếm 61.3% unique skills) đang thống trị 80% nhu cầu thị trường.
   - ⚠️ Chẩn đoán: Dữ liệu kỹ năng có tính thưa thớt (Sparsity) cao. Nhiều kỹ năng ngách (long-tail) chỉ xuất hiện 1-2 lần.


C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_18688\1819011074.py:65: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha="right", fontsize=10)


✅ Đã lưu Biểu đồ 6 (Pareto Skills) tại: D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vietnamworks\03_skills_pareto_distribution.png
✅ Đã lưu Biểu đồ 7 (Bias & Balance) tại: D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vietnamworks\04_bias_and_class_balance.png


# Headmap

In [23]:

def plot_correlation_heatmap(df, save_path):
    """
    Trích xuất các biến định lượng, tính toán ma trận tương quan Spearman
    và vẽ Heatmap (bỏ qua nửa tam giác trên để tránh trùng lặp).
    """
    print("--- VẼ MA TRẬN TƯƠNG QUAN HẠNG SPEARMAN ---")
    
    # 1. TRÍCH XUẤT VÀ TẠO CÁC BIẾN ĐỊNH LƯỢNG (NUMERIC FEATURES)
    df_num = pd.DataFrame()
    
    # Lương: Chuyển đổi về Triệu VNĐ và thay thế mức lương 0 (Thỏa thuận) thành NaN
    if 'salaryMin' in df.columns:
        df_num['Salary_Min_M'] = df['salaryMin'].replace(0, np.nan) / 1_000_000
    if 'salaryMax' in df.columns:
        df_num['Salary_Max_M'] = df['salaryMax'].replace(0, np.nan) / 1_000_000
        
    # Kinh nghiệm: Số năm tối thiểu
    if 'yearsOfExperience' in df.columns:
        df_num['Years_of_Experience'] = pd.to_numeric(df['yearsOfExperience'], errors='coerce')
        
    # Tạo biến đếm số lượng Kỹ năng (Dựa vào độ dài của list trong cột keywords)
    if 'keywords' in df.columns:
        df_num['Total_Skills_Count'] = df['keywords'].apply(
            lambda x: len(x) if isinstance(x, list) else 0
        )
        
    # Tạo biến đếm độ dài Mô tả công việc (JD Word Count)
    if 'description_list' in df.columns:
        df_num['JD_Word_Count'] = df['description_list'].apply(
            lambda x: sum(len(str(item).split()) for item in x) if isinstance(x, list) else 0
        )

    # 2. TÍNH TOÁN MA TRẬN TƯƠNG QUAN SPEARMAN
    # Sử dụng Spearman thay vì Pearson vì dữ liệu lương và kinh nghiệm không phân phối chuẩn (bị lệch)
    corr_matrix = df_num.corr(method="spearman")
    
    # 3. VẼ HEATMAP TRỰC QUAN
    plt.figure(figsize=(10, 8))
    
    # Tạo mask để ẩn nửa tam giác phía trên đường chéo chính (tránh rối mắt)
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    # Vẽ heatmap bằng Seaborn
    sns.heatmap(
        corr_matrix, 
        mask=mask, 
        annot=True,          # Hiển thị số
        fmt=".2f",           # Lấy 2 chữ số thập phân
        cmap="coolwarm",     # Bảng màu Xanh (Âm) - Đỏ (Dương)
        vmin=-1, vmax=1,     # Giới hạn thang đo từ -1 đến 1
        linewidths=1, 
        cbar_kws={"shrink": 0.8},
        annot_kws={"size": 11, "weight": "bold"}
    )
    
    plt.title("Ma Trận Tương Quan Hạng Spearman\nGiữa Các Thuộc Tính Định Lượng (VietnamWorks)", 
              fontsize=14, fontweight='bold', pad=20)
    
    # Chỉnh góc nghiêng cho tên cột để dễ đọc
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.yticks(rotation=0, fontsize=11)
    
    plt.tight_layout()
    
    # 4. LƯU FILE
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Hoàn tất! Đã lưu Heatmap tương quan tại: {save_path}")
    return corr_matrix

# ==========================================
# THỰC THI PIPELINE VẼ BIỂU ĐỒ 5
# ==========================================
save_path_chart_5 = r"D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vietnamworks\05_spearman_correlation_matrix.png"

# Trả về corr_matrix để có thể print ra console xem nhanh
correlation_table = plot_correlation_heatmap(df_vietnamworks, save_path_chart_5)
display(correlation_table)

--- VẼ MA TRẬN TƯƠNG QUAN HẠNG SPEARMAN ---
✅ Hoàn tất! Đã lưu Heatmap tương quan tại: D:\PYTHON\Data mining\VN-IT-Job-Mining\reports\figures\vietnamworks\05_spearman_correlation_matrix.png


,Salary_Min_M,Salary_Max_M,Years_of_Experience,Total_Skills_Count,JD_Word_Count
Salary_Min_M,1.000000,0.960564,0.090718,0.142644,0.068557
Salary_Max_M,0.960564,1.000000,0.075803,0.141152,0.043689
Years_of_Experience,0.090718,0.075803,1.000000,0.101848,0.159000
Total_Skills_Count,0.142644,0.141152,0.101848,1.000000,0.028663
JD_Word_Count,0.068557,0.043689,0.159000,0.028663,1.000000
